In [1]:
import numpy as np
import cudaq

# CHSH Game Rule

Randomly assign question $x$ and $y$ to be either 0 or 1, and send them to Alice and Bob respectively (They don't know each other's question).

Alice and Bob will then each output a bit, $a$ and $b$ respectively. They win if the following condition is satisfied:

$$
\begin{array}{ccc}
(x,y) & \text{win} & \text{lose} \\[1mm]\hline
\rule{0mm}{4mm}(0,0) & a = b & a \neq b \\[1mm]
(0,1) & a = b & a \neq b \\[1mm]
(1,0) & a = b & a \neq b \\[1mm]
(1,1) & a \neq b & a = b
\end{array}
$$

or equivalently, they win if $a \oplus b = x \land y$.

In [2]:
class CHSH_agent:
    def __init__(self):
        self.total_plays = 0
        self.wins = 0

    def create_new_game(self):
        self.total_plays += 1
        self.xy = np.random.randint(0, 2, size=2)
        return self.xy
    
    def play(self, a, b):
        x, y = self.xy
        self.wins += ((a ^ b) == (x & y)) ## FILL CODE HERE ##

# Classical Approach

Let's try 2 approaches to solve this problem classically.

### 1. Deterministic Strategy

From the table, if both alice and bob always output 0 or 1, they will win 3 out of 4 times.

In [3]:
deterministic_agent = CHSH_agent()
for _ in range(100000):
    deterministic_agent.create_new_game()
    deterministic_agent.play(0, 0)
print(f"Deterministic strategy wins: {deterministic_agent.wins}/{deterministic_agent.total_plays} = {deterministic_agent.wins/deterministic_agent.total_plays:.2f}")

Deterministic strategy wins: 74718/100000 = 0.75


### 2. Random Strategy

Now it's your time to try! Can you do better than ~75%?

Ans: No, I can't.

In [4]:
random_agent = CHSH_agent()
for _ in range(100000):
    random_agent.create_new_game()
    ## EDIT CODE HERE ##
    alice_output = np.random.choice([0, 1], p=[0.4, 0.6])
    bob_output = np.random.choice([0, 1], p=[0.75, 0.25])
    ####################
    random_agent.play(alice_output, bob_output)
print(f"Random strategy wins: {random_agent.wins}/{random_agent.total_plays} = {random_agent.wins/random_agent.total_plays:.2f}")

Random strategy wins: 47556/100000 = 0.48


# Quantum Approach

Details and proof can be found in [IBM course](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/entanglement-in-action/chsh-game)

You may want to consult with [CUDA-Q operation doc](https://nvidia.github.io/cuda-quantum/latest/using/examples/quantum_operations.html)

In short,

1. Prepare a Bell state $|\phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ shared between Alice and Bob.
2. Alice applies $R_y(0)$ if $x=0$ and $R_y(-\pi/2)$ if $x=1$.
3. Bob applies $R_y(-\pi/4)$ if $y=0$ and $R_y(\pi/4)$ if $y=1$.

In [5]:
@cudaq.kernel
def quantum_strategy(x: int, y: int):
    alice_qbit = cudaq.qubit()
    bob_qbit = cudaq.qubit()

    # Prepare Bell state
    ## FILL CODE HERE ##
    h(alice_qbit)
    x.ctrl(alice_qbit, bob_qbit)
    ####################

    # Apply rotations based on Alice's question
    if x == 0:
        ## FILL CODE HERE ##
        ry(0, alice_qbit)
        ####################
    else:
        ## FILL CODE HERE ##
        ry(-np.pi / 2, alice_qbit)
        ####################

    # Apply rotations based on Bob's question
    if y == 0:
        ## FILL CODE HERE ##
        ry(-np.pi / 4, bob_qbit)
        ####################
    else:
        ## FILL CODE HERE ##
        ry(np.pi / 4, bob_qbit)
        ####################

    # Alice measures her qubit
    ## FILL CODE HERE ##
    mz(alice_qbit)
    ####################

    # Bob measures his qubit
    ## FILL CODE HERE ##
    mz(bob_qbit)
    ####################

In [6]:
quantum_agent = CHSH_agent()
for _ in range(10000):
    xy = quantum_agent.create_new_game()
    x, y = xy
    result = cudaq.sample(quantum_strategy, int(x), int(y),shots_count=1)
    alice_output, bob_output = [int(_) for _ in result.most_probable()]
    quantum_agent.play(alice_output, bob_output)
print(f"Quantum strategy wins: {quantum_agent.wins}/{quantum_agent.total_plays} = {quantum_agent.wins/quantum_agent.total_plays:.2f}")  

Quantum strategy wins: 8577/10000 = 0.86


## Question: The exact winning probability is? (pls visit the IBM course for the proof)

#### ANS: $\cos^2(\pi/8) = \frac{2 + \sqrt{2}}{4}$
